In [ ]:
from quspin.operators import hamiltonian # 用于在给定的基（basis）上构建哈密顿量算符(或其他物理观测量)；
from quspin.basis import spin_basis_1d # 用于创建一维自旋-1/2链的希尔伯特空间基；
import numpy as np 
import matplotlib.pyplot as plt  # 用于结果可视化

In [ ]:
######## 关于一维海森堡XXX模型的建立与计算(只考虑最近邻相互作用、周期边界条件)
##### 一.参数设置
L = 20
J = 1.0

##### 二.定义基(包括各种对称性的考虑),以下是函数spin_basis_1d()完整表示
basis = spin_basis_1d(
    L=L,         # 必填(int)：一维链长度；
    S="1/2",     # 可选(str或float,默认自旋-1/2)：每个格点上的自旋量子数；
    pauli=True, # 可选：pauli=False(默认):使用物理自旋算符表象,例如,"x"对应 Sx,"zz"对应SizSjz; pauli=True使用Pauli表象,此时"x"对应σx=2Sx；
    Nup=L//2,   # 可选(int 或 list of int)：由U1对称性,通过指定系统自旋向上的总数Nup,来确定总自旋Sz量子数的子空间(比如L//2表示一半自旋向上)；
    kblock=None,# 可选(int)：指定动量块即波矢k的取值(平移对称性的量子数，对应动量2πk/L),要求系统具有周期性边界即平移对称性(a=1且pblock=1)；
    pblock=None,   # 可选(int,取值1或-1)：指定空间反射(宇称)对称性的量子数，1对应偶宇称、-1对应奇宇称
    zblock=None,# 可选(int,取值1或-1)：表示指定自旋反演对称性的量子数,1对应偶对称性(注意：None是默认表示没有,并且Nup、kblock、pblock也同理)；
    a=1         #可选(int,默认为1)：平移对称性下原胞格点数。当 a>1 时，系统被视为具有 a-site 的晶胞结构；
)
# 返回basis：是一个基对象,包含了所有必要信息的容器(不是单纯基向量列表)，比如格点数N、每个格点维数sps、在给定对称性约束下当前基的维度Ns。

##### 三.哈密顿量的构建(周期边界条件)：
### 3.1构建耦合列表：周期性边界,耦合强度J(其中[J,i,j]为每个格点i生成一个包含三个元素的子列表,而J表示耦合强度,i、j表示当前格点与相邻格点索引)；
bond_list_xy = [[J/8, i, (i+1)%L] for i in range(L)] # 自旋xy对应的耦合列表(将Sx与Sy用S+与S-表示则所有矩阵都是实矩阵，但会多出现1/2)；
bond_list_zz = [[J/4, i, (i+1)%L] for i in range(L)]  # 自旋z对应的耦合列表
# 注意：j=(i+1)%L(%为取余)为周期边界条件(因为对最后格点i=L-1经过该操作得到j=0,即L-1与0格点相邻)，而开放边界条件为j=i+1。

### 3.2构建哈密顿量
static = [["+-", bond_list_xy], ["-+", bond_list_xy], ["zz", bond_list_zz]] # 定义哈密顿量的静态部分(不随时间变化的算符)
dynamic = [] # 定义哈密顿量的动态部分(随时间变化的算符),空列表表示哈密顿量中没有随时间变化的部分。
H = hamiltonian(  # 使用QuSpin的hamiltonian构造函数来得到哈密顿量对象，其矩阵形式是自动使用稀疏矩阵格式存储哈密顿量(但H不只是矩阵)；
    static, 
    dynamic, 
    basis=basis, 
    dtype=np.float64,# 必填：dtype=np.float64为设置矩阵的数据类型为64位浮点数,这确保了数值计算的精度和效率,特别适用于实对称哈密顿量(如自旋)；
    check_symm=True, # 可选(bool布尔类型)：check_symm=True(默认)表示验证运算符是否与基底对称性兼容；check_symm=False表示禁用；
    check_pcon=True, # 可选(bool布尔类型)：check_pcon=True(默认)表示检查运算符是否保持粒子数守恒；check_pcon=False表示禁用；
    check_herm=True # 可选(bool布尔类型)：check_herm=True(默认)表示验证运算符是否为厄米算符；check_herm=False表示禁用；
) 
# 注意:返回H是一个智能的、功能齐全的哈密顿量计算引擎(包含了数据(矩阵)和操作(方法)的完整包),能直接执行几乎所有常见的量子多体计算任务,见下面。

### 3.3查看返回的H的特性
print('='*80)
print(f"系统尺寸 L = {L}")
print("H 对象的类型:", type(H))
print("希尔伯特空间维度:", H.Ns) # 通过H对象访问其basis的属性
print("矩阵形状:", H.shape) # H作为运算符的矩阵形状

##### 四.具体计算：
### 4.1能量(Energy)
## 4.1.1计算基态与基态能量
E_gs, V_gs = H.eigsh(k=1, which='SA') # eigsh为稀疏矩阵对角化法，k参数要求方法计算前k个本征值，which='SA'代表取前k个最小本征值与本征态
E_gs = E_gs[0] # 因为返回的E_gs是数组(哪怕只有一个元素)，所以需通过E_gs[0]得到具体数字
V_gs = V_gs[:, 0] # 注意：psi_gs是形状为(Ns,1)的二维矩阵(Ns为希尔伯特空间的维数),而psi_gs[:,0]是形状为(Ns,)的一维数组(以后计算以数组为主)
print('='*80)
print('系统能量：')
print()
print(f"计算基态能量: {E_gs:.10f}")
print(f"每格点能量: {E_gs/L:.10f}")
print()

# 与热力学极限值对比
E0_infinite = (1/4 - np.log(2)) # 约 -0.443147
print(f"热力学极限每格点能量: {E0_infinite:.6f}")
print(f"有限尺寸修正: {E_gs/L - E0_infinite:.6f}")

## 4.1.2对于较小系统，可以计算其全部本征值，并作图
print('-'*80)
if H.Ns <= 100:  # 仅当希尔伯特空间维度H.Ns较小时执行
    E_full = H.eigvalsh()
    print(f"\n完整能谱（前5个）: {E_full[:5]}")
    
    # 绘制能级分布
    plt.figure(figsize=(8, 4))
    plt.plot(E_full, 'bo-', linewidth=2, markersize=8) # 'bo-'中b表示蓝色(blue)、o表示每个数据点用圆圈标记、- 表示用实线连接数据点。
    plt.xlabel('Energy level index', fontsize=12)
    plt.ylabel('Energy', fontsize=12)
    plt.title('Energy distribution of the Heisenberg model', fontsize=15)
    plt.grid(True, alpha=0.3)
    plt.show()

### 4.2 自旋平均值(Average value of spin or Expectation value of spin)
# 计算每个格点的自旋基态平均值(其他态同理)。注意：由于pauli=True，算符"z"对应pauli算符σ_z = 2S_z，因此期望值需除以2得到物理自旋算符S_z
S_z_avg = []  # 创建存储每个格点的S_z平均值
print('='*80)
print('每个格点的自旋平均值：')
print()

for i in range(L):
    # 构建第i个格点的σ_z算符(与哈密顿量的构建完全类似)
    static_σz = [["z", [[1.0, i]]]] # 自旋算符的静态部分,其中"z"表示pauli算符σ_z，[[1.0, i]]表示作用在格点i且耦合强度1.0
    dynamic_σz = [] # 自旋算符的动态部分(此处为空)
    σ_z_i = hamiltonian(static_σz, dynamic_σz, basis=basis, dtype=np.float64, check_symm=False, check_pcon=False, check_herm=False)
    
    # 计算基态期望值，并转换为自旋算符S_z（除以2）
    exp_val = σ_z_i.expt_value(V_gs).real  # σ_z_i.expt_value为σ_z_i的期望值计算函数,V_gs为上面得到的基态,real表示pauli算符期望值为实数
    S_z_physical = exp_val / 2.0  # 自旋S_z算符期望值
    S_z_avg.append(S_z_physical)
    print(f"格点 {i} 的 S_z 平均值: {S_z_physical:.6f}")

# 可视化S_z平均值
plt.figure(figsize=(8, 4))
plt.plot(range(L), S_z_avg, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Site index i', fontsize=12)
plt.ylabel('$\\langle S_i^z \\rangle$', fontsize=12) 
# 注：$符号用作数学模式的分界符，即表明$...$之间的内容是需要用特殊规则排版的数学公式(以LaTeX为标准)。其中\\langle ...\\rangle表示期望值
plt.title('Average spin per site', fontsize=15)
plt.grid(True, alpha=0.3)
plt.show()

### 4.3 自旋关联函数(Spin correlation function)
# 在平移对称性体系下(此时关联函数只是相对距离的函数)取周期边界条件，以自旋算符Sz-Sz、两格点为(i=0,j)且态取基态为例计算(其他关联函数同理)
ref_site = 0  # 参考格点i=0索引
correlations = [] # 存储关联值
distances = [] # 存储距离
print('='*80)
print('自旋关联函数：')
print()

# 平移对称性下关联函数只是相对距离的函数,故在周期边界条件下,格点i=0到格点j的距离d从1到L/2(因为i与j=L-1是相邻的);若为开放边界条件,则d从1到L-1
for d in range(1, L//2 + 1): 
    j = (ref_site + d) % L # 目标格点索引(取余%操作是确保当ref_site + d >= L时，会循环回到链的开头)
    
    # 构建σ_i^z -σ_j^z算符，注意pauli=True时"zz"对应σ_i^z - σ_i^z = 4 S_i^z - S_i^z
    static_zz = [["zz", [[1.0, ref_site, j]]]]  # 关联函数算符的静态部分,其中耦合强度1.0,"zz"表示σ_i^z-σ_j^z
    dynamic_zz = [] # 关联函数算符的动态部分
    S_zz_ij = hamiltonian(static_zz, dynamic_zz, basis=basis, dtype=np.float64, check_symm=False, check_pcon=False, check_herm=False)
    
    # 计算基态期望值，并转换为自旋S_z·S_z关联（除以4）
    exp_val_zz = S_zz_ij.expt_value(V_gs).real
    S_zS_z_physical = exp_val_zz / 4.0  # σ_i^z σ_j^z = 4 S_i^z S_j^z → S_i^z S_j^z = <σ_i^z σ_j^z>/4
    correlations.append(S_zS_z_physical)
    distances.append(d)
    print(f"格点 {ref_site} 与格点 {j} (距离 d={d}) 的 S_z·S_z 关联: {S_zS_z_physical:.6f}")

# 可视化关联函数
plt.figure(figsize=(8, 4))
plt.plot(distances, correlations, 'bs-', linewidth=2, markersize=8) # 's'表示数据点取正方形(square)
plt.xlabel('Distance d', fontsize=12)
plt.ylabel('$\\langle S_0^z \cdot S_d^z \\rangle$', fontsize=12)
plt.title('Spin correlation function (reference site 0)', fontsize=15)
plt.grid(True, alpha=0.3)
plt.show()

### 4.4 纠缠熵(Entanglement entropy)
# 4.4.1将系统分为子系统A（前L/2个格点）和余下部分，计算基态的纠缠熵
sub_sys_A = range(L//2)  # 子系统A包含格点0到L/2-1
ent_entropy_dict = basis.ent_entropy(V_gs, sub_sys_A=sub_sys_A) # basis.ent_entropy函数用来计算态V_gs、子系统sub_sys_A下的纠缠熵,并返回字典
Sent_A = ent_entropy_dict['Sent_A'] #字典ent_entropy_dict结构:'Sent_A'(子系统A的纠缠熵);'Sent_B'(补集B的纠缠熵);'p_A'(约化密度矩阵本征值)等
print('='*80)
print('系统纠缠熵：')
print()
print(f"子系统A（前{L//2}个格点）的纠缠熵: {Sent_A:.6f}")

# 4.4.2可选：计算不同子系统的基态纠缠熵以观察尺度行为。例如，变化子系统大小并计算基态纠缠熵
subsystem_sizes = range(1, L) # 定义子系统大小的范围，从 1 到 L−1
entropies = [] # 储存所有子系统大小的纠缠熵值
print('-'*80)
print("不同尺寸子系统A的基态纠缠熵：")
print()
    
for size in subsystem_sizes:
    sub_sys = range(size)
    ent_dict = basis.ent_entropy(V_gs, sub_sys_A=sub_sys)
    Sent_A = ent_dict['Sent_A']
    entropies.append(Sent_A)
    print(f"子系统A（前{len(sub_sys)}个格点）的纠缠熵: {Sent_A:.6f}")

# 可视化纠缠熵
plt.figure(figsize=(8, 4))
plt.plot(subsystem_sizes, entropies, 'bo-', linewidth=2)
plt.xlabel('Subsystem size', fontsize=12)
plt.ylabel('Entanglement entropy', fontsize=12)
plt.title('Entanglement entropy vs. subsystem size', fontsize=15) 
plt.grid(True, alpha=0.3)
plt.show()



In [ ]:
#--------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------           
########## 额外扩展一、开放边界条件下一维海森堡模型的建立
# 与周期边界条件的区别：1.在定义基basis中，一定要令kblock=None(可以不写表示默认None)，而其他对称性根据体系来判断；
#                       2.哈密顿量的构建步骤中，在耦合列表处将(i+1)%L的%L去掉，并且把for i in range(L)中L改成L-1。

print('='*80)
print('='*80)
##### 一.参数设置
L1 = 8
J = 1.0

##### 二.定义基(包括各种对称性的考虑)s
basis_1 = spin_basis_1d(L=L1, pauli=True, Nup=L1//2,   
    kblock=None, # 特别注意：开放边界条件下kblock=None(可以不写表示默认为None)；
    pblock=None, 
    zblock=None,
    a=1         
)

##### 三.哈密顿量的构建(开放边界条件)：
### 3.1构建耦合列表。特别注意：开放边界条件下，除了去除周期边界条件中(i+1)%L的%L，还要把for i in range(L)中L改成L-1。
bond_list_xy = [[J/8, i, (i+1)] for i in range(L1-1)] # 自旋xy对应的耦合列表(将Sx与Sy用S+与S-表示则所有矩阵都是实矩阵，但会多出现1/2)；
bond_list_zz = [[J/4, i, (i+1)] for i in range(L1-1)]  # 自旋z对应的耦合列表

### 3.2构建哈密顿量
static = [["+-", bond_list_xy], ["-+", bond_list_xy], ["zz", bond_list_zz]] # "+-"表示S+S-
dynamic = [] 
H_1 = hamiltonian(static, dynamic, basis=basis, dtype=np.float64, check_symm=False) 

### 3.3查看返回的H的特性
print()
print(f"系统尺寸 L = {L1}")
print("H 对象的类型:", type(H))
print("希尔伯特空间维度:", H.Ns) # 通过H对象访问其basis的属性
print("矩阵形状:", H.shape) # H作为运算符的矩阵形状
print()

#### 四.物理量计算验证
# 计算基态能量
E_gs_1, V_gs_1 = H_1.eigsh(k=1, which='SA')
print(f"系统基态能量: {E_gs_1[0]:.8f}")
print(f"系统基态: {V_gs_1}")
print()

# 计算能谱（前几个本征值）
E_vals_1, V_vecs_1 = H_1.eigsh(k=5, which='SA')
print("前5个本征值:")
for i, E in enumerate(E_vals_1):
    print(f"  E_{i} = {E:.6f}")

In [ ]:
#--------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------           
########## 额外扩展二：考虑次近邻相互作用的一维海森堡模型的建立

print('='*80)
print('='*80)
#### 一.参数设置
L2 = 8
J1 = 1.0  # 最近邻相互作用强度
J2 = 0.5  # 次近邻相互作用强度

#### 二.定义基（S^z=0 子空间）
basis_2 = spin_basis_1d(L=L2, pauli=True, Nup=L2//2, kblock=0, pblock=1)

#### 三.哈密顿量的构建：
## 3.1构建耦合列表(周期边界条件)
nn_bond_list = [[J1/4, i, (i+1)%L2] for i in range(L2)] # 最近邻耦合列表（i和i+1）
nnn_bond_list = [[J2/4, i, (i+2)%L2] for i in range(L2)] # 次近邻耦合列表（i和i+2）

## 3.2构建哈密顿量
static = [
    ["xx", nn_bond_list], # 最近邻海森堡相互作用
    ["yy", nn_bond_list], 
    ["zz", nn_bond_list],
    ["xx", nnn_bond_list],# 次近邻海森堡相互作用
    ["yy", nnn_bond_list], 
    ["zz", nnn_bond_list]
]
dynamic = []
H_2 = hamiltonian(static, dynamic, basis=basis_2, dtype=np.float64)

## 3.3查看哈密顿量特性
print()
print(f"系统尺寸 L = {L2}")
print("H 对象的类型:", type(H_2))
print("希尔伯特空间维度:", H_2.Ns)
print("矩阵形状:", H_2.shape)
print("最近邻相互作用强度 J1 =", J1)
print("次近邻相互作用强度 J2 =", J2)
print()

#### 四.物理量计算验证
# 计算基态能量
E_gs_2, V_gs_2 = H_2.eigsh(k=1, which='SA')
print(f"系统基态能量: {E_gs_2[0]:.8f}")
print(f"系统基态: {V_gs_2}")
print()

# 计算能谱（前几个本征值）
E_vals_2, V_vecs_2 = H_2.eigsh(k=5, which='SA')
print("前5个本征值:")
for i, E in enumerate(E_vals_2):
    print(f"  E_{i} = {E:.6f}")